In [64]:
data = [
    ("i am a student", "nenu oka student "),
    ("how are you", "ela unnavu"),
    ("i love machine learning", "naku machine learning nate istam"),
    ("good morning", "shubodhyam"),
    ("thank you", "dhanyavadhalu"),
    ("see you later", "malli kaludham"),
    ("what is your name", "ni peru enti"),
    ("where are you going", "nuvvu ekkadiki velthunnavu"),
    ("i like coffee", "naku coffee ante istam"),
    ("welcome", "swagatham")
]

In [65]:
# Import necessary libraries 
import tensorflow as tf 
import numpy as np 
from tensorflow.keras.layers import TextVectorization,Embedding,Dense,LayerNormalization,MultiHeadAttention
from tensorflow.keras import Model

In [66]:
# Seperate Input and Output sentences
english_sentences=[x [0] for x in data] # List comprehension to extract English 
# add START and ENd tokens 
telugu_sentences=[
    "Start " + x[1] + " End" for x in data
]


In [67]:
# tokenization
vocab_size=1000 # keep a maximum of 1000 unique words in vocabulary 
sequence_length=20 # maximum length of each sentence 

# Tokenize english sentences 
source_vectorization=TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length
)
# Tokenize French sentences
target_vectorization=TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length
)
# This is where learning happens.(Adapt the tokens to the data)
source_vectorization.adapt(english_sentences)
target_vectorization.adapt(telugu_sentences)


In [68]:
# Convert text into numbers 
encoder_inputs=source_vectorization(english_sentences)

target_tokens=target_vectorization(telugu_sentences)


In [69]:
# Prepare decoder inputs and output 
decoder_inputs=target_tokens[:, :-1]
decoder_outputs=target_tokens[:, 1:]


In [70]:
# Positional encoding
class PositionalEmbedding(tf.keras.layers.Layer):
    # Constructor 
    def __init__(self,sequence_length,vocab_size,embed_dim):
        # Call the parent constructor
        super().__init__()
        # Token Embedding 
        self.token_embedding=Embedding(input_dim=vocab_size,output_dim=embed_dim)
        # Positional Embedding
        self.position_embedding=Embedding(input_dim=sequence_length,output_dim=embed_dim)

        # Store sequence length for later use 
        self.sequence_length=sequence_length
    
    # Call method to compute the output of the layer
    def call(self,inputs):
        # Get the length of the input sequence 
        length=tf.shape(inputs)[-1]
        # Create position numbers 
        positions=tf.range(start=0,limit=length,delta=1) 
         
        # Convert words to embeddings 
        embedded_tokens=self.token_embedding(inputs)
        # Convert positions to embeddings
        embedded_positions=self.position_embedding(positions)
        #Add both embeddings together
        return embedded_tokens + embedded_positions



In [71]:
# Encoder block 
# Create a custom Encoder Layer
class TransformerEncoder(tf.keras.layers.Layer):
    # Constructor 
    # embed_dim:128,dense_dim:512,num_heads:4
    # num_heads: Head1-->Grammer,Head2-->Context,Head3-->Relationships,Head4-->Meaning 

    def __init__(self,embed_dim,dense_dim,num_heads):
        super().__init__()
        # Multi-Head Attention layer
        self.attention=MultiHeadAttention(num_heads=num_heads,key_dim=embed_dim)
        # Feed forward network
        # Attention mixes information .
        # FFN learns complex features
        # I love ai (Input)---->Attention---->Ai is related to love---->FFN---->(Ai is the object being loved)I love AI(output)
        self.dense_proj=tf.keras.Sequential([
            Dense(dense_dim,activation="relu"),
            Dense(embed_dim)
        ])
        # Layer Normalization 
        self.layernorm1=LayerNormalization()
        self.layernorm2=LayerNormalization()

    def call(self,inputs):
         # Self Attention
        attention_output=self.attention(inputs,inputs)
         #Residual connection + LayerNorm
        proj_input=self.layernorm1(inputs + attention_output)
        # Feed forward network
        proj_output=self.dense_proj(proj_input)

         # Second Residual Connection + LayerNorm 
        return self.layernorm2(proj_input + proj_output)
            


In [72]:
# Decoder Block 
class TransformerDecoder(tf.keras.layers.Layer):
    def __init__(self,embed_dim,dense_dim,num_heads):
        super().__init__()
    
        self.self_attention=MultiHeadAttention(num_heads=num_heads,key_dim=embed_dim)
        self.cross_attention=MultiHeadAttention(num_heads=num_heads,key_dim=embed_dim)
        # Feed forward network
        self.ffn=tf.keras.Sequential([
            Dense(dense_dim,activation="relu"),
            Dense(embed_dim)
        ])
        # Layer Normalization 
        self.layernorm1=LayerNormalization()
        self.layernorm2=LayerNormalization()
        self.layernorm3=LayerNormalization()
    
    def call(self,inputs,encoder_outputs):
        # Masked self attention
        attention_output=self.self_attention(
            query=inputs,
            value=inputs,
            key=inputs,
            use_causal_mask=True
        )
        out1=self.layernorm1(inputs + attention_output)
        
        # Cross attention
        attention_output2=self.cross_attention(
           out1,
           encoder_outputs
        )
        out2=self.layernorm2(out1 + attention_output2)

        # Feed forward network
        ffn_output=self.ffn(out2)
        return self.layernorm3(out2 + ffn_output)

In [73]:
# Build Complete transformer 
embed_dim=128 
dense_dim=256 
num_heads=4

# Encoder input layer 
encoder_input=tf.keras.Input(
    shape=(None,),dtype="int64"
)
x= PositionalEmbedding(sequence_length,vocab_size,embed_dim)(encoder_input)

#Encoder block 
encoder_output=TransformerEncoder(embed_dim,dense_dim,num_heads)(x)

# Decoder block 
decoder_input=tf.keras.Input(
    shape=(None,),dtype="int64"
)

x= PositionalEmbedding(sequence_length,vocab_size,embed_dim)(decoder_input)

x=TransformerDecoder(embed_dim,dense_dim,num_heads)(x,encoder_output)

decoder_output=Dense(vocab_size,activation="softmax")(x)

#create model
transformer=Model([encoder_input,decoder_input],decoder_output)




In [74]:
# Compile and train 
transformer.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

transformer.fit(
    [encoder_inputs,decoder_inputs],
    decoder_outputs,
    batch_size=2,
    epochs=10
)

Epoch 1/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 9s 22ms/step - accuracy: 0.6316 - loss: 4.5584  
Epoch 2/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.8158 - loss: 2.4265
Epoch 3/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.8158 - loss: 1.7233
Epoch 4/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.8158 - loss: 1.2888
Epoch 5/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.8316 - loss: 0.9814
Epoch 6/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.8474 - loss: 0.8029
Epoch 7/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.8368 - loss: 0.6840
Epoch 8/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.8789 - loss: 0.5858
Epoch 9/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.8737 - loss: 0.5021
Epoch 10/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.9053 - loss: 0.4191


In [75]:
# Prediction
 
test_sentence = ["i like coffee"]
 
encoder_input_test = source_vectorization(
    test_sentence
)
 
start_sentence = "start"
 
decoded_sentence = start_sentence
 
for i in range(10):
 
    tokenized_target = target_vectorization(
        [decoded_sentence]
    )
 
    predictions = transformer.predict(
        [
            encoder_input_test,
            tokenized_target
        ],
        verbose=0
    )
 
    sampled_token_index = np.argmax(
    predictions[0, len(decoded_sentence.split()) - 1, :]
)
 
    index_lookup = dict(
        zip(
            range(
                len(
                    target_vectorization.get_vocabulary()
                )
            ),
            target_vectorization.get_vocabulary()
        )
    )
 
    sampled_token = index_lookup[
        sampled_token_index
    ]
 
    decoded_sentence += " " + sampled_token
 
    if sampled_token == "end":
        break
 
print(decoded_sentence)
 

start naku coffee ante end
